# 01 - Extracción de datos Gold

Este notebook extrae las vistas de la capa **Gold** desde Supabase y las guarda como archivos `.csv` en `data/processed`.

Vistas utilizadas:
- `gold.modelo_semaforo_asesor_mes`
- `gold.modelo_predictivo_semaforo`
- `gold.puntos_mejora_asesor`


In [1]:
from pathlib import Path
import sys

import pandas as pd

# Detectar raíz del proyecto de forma segura
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / ".secrets").exists():
    ROOT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / ".secrets").exists():
    ROOT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "No se encontró la carpeta .secrets. "
        "Ejecuta el notebook desde la raíz del proyecto o desde la carpeta notebooks."
    )

SECRETS_DIR = ROOT_DIR / ".secrets"
DATA_DIR = ROOT_DIR / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SECRETS_DIR))

from db_config import get_connection

ROOT_DIR, DATA_DIR

(WindowsPath('d:/INSTITUTO CONTINENTAL/4. CICLO - 4/BIG DATA APLICADA/PP2B_SATDPO'),
 WindowsPath('d:/INSTITUTO CONTINENTAL/4. CICLO - 4/BIG DATA APLICADA/PP2B_SATDPO/data/processed'))

## 1. Validación rápida de vistas Gold

Antes de extraer los datos, validamos que las vistas Gold existan y tengan registros.


In [2]:
conn = get_connection()

query_validacion = """
select 'gold.modelo_semaforo_asesor_mes' as vista, count(*) as filas
from gold.modelo_semaforo_asesor_mes
union all
select 'gold.modelo_predictivo_semaforo' as vista, count(*) as filas
from gold.modelo_predictivo_semaforo
union all
select 'gold.puntos_mejora_asesor' as vista, count(*) as filas
from gold.puntos_mejora_asesor;
"""

df_validacion = pd.read_sql(query_validacion, conn)
conn.close()

df_validacion

C:\Users\Steph\AppData\Local\Temp\ipykernel_6756\2949662675.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_validacion = pd.read_sql(query_validacion, conn)


,vista,filas
0,gold.modelo_semaforo_asesor_mes,1680
1,gold.modelo_predictivo_semaforo,1620
2,gold.puntos_mejora_asesor,10080


## 2. Extraer dataset predictivo

Este archivo será usado para el entrenamiento del modelo.  
Incluye el semáforo actual y la variable objetivo `semaforo_mes_siguiente`.


In [3]:
conn = get_connection()

query_predictivo = """
select *
from gold.modelo_predictivo_semaforo
order by dni, periodo;
"""

df_predictivo = pd.read_sql(query_predictivo, conn)
conn.close()

df_predictivo.shape

C:\Users\Steph\AppData\Local\Temp\ipykernel_6756\2725423198.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_predictivo = pd.read_sql(query_predictivo, conn)


(1620, 37)

In [4]:
df_predictivo.head()

,dni,asesor,area,sede,fecha_ingreso,estado_trabajador,anio,mes,periodo,aprobacion_inmediata_pct,...,alerta_tmo,alerta_nps,alerta_calidad,alerta_tipificacion,score_final,semaforo_actual,tmo_original_bronze,semaforo_mes_siguiente,score_mes_siguiente,periodo_siguiente
0,40107223,María Rodríguez Morales,Gestión de Reportes,Los Olivos,2021-08-03,Cesado,2024,01,2024-01,45.0,...,Crítico,Adecuado,Crítico,Crítico,53.69,Rojo,6.87,Rojo,69.06,2024-02
1,40107223,María Rodríguez Morales,Gestión de Reportes,Los Olivos,2021-08-03,Cesado,2024,02,2024-02,55.0,...,Crítico,Adecuado,Crítico,Crítico,69.06,Rojo,6.67,Ámbar,71.91,2024-03
2,40107223,María Rodríguez Morales,Gestión de Reportes,Los Olivos,2021-08-03,Cesado,2024,03,2024-03,50.0,...,Crítico,Adecuado,Crítico,Por mejorar,71.91,Ámbar,6.94,Rojo,65.96,2024-04
3,40107223,María Rodríguez Morales,Gestión de Reportes,Los Olivos,2021-08-03,Cesado,2024,04,2024-04,45.0,...,Crítico,Adecuado,Crítico,Por mejorar,65.96,Rojo,6.98,Rojo,24.11,2024-05
4,40107223,María Rodríguez Morales,Gestión de Reportes,Los Olivos,2021-08-03,Cesado,2024,05,2024-05,45.0,...,Crítico,Crítico,Crítico,Crítico,24.11,Rojo,6.84,Ámbar,85.85,2024-06


In [5]:
if "semaforo_mes_siguiente" in df_predictivo.columns:
    display(df_predictivo["semaforo_mes_siguiente"].value_counts(dropna=False))
else:
    print("No existe la columna semaforo_mes_siguiente en el dataset predictivo.")

semaforo_mes_siguiente
Rojo     1068
Ámbar     552
Name: count, dtype: int64

In [6]:
output_predictivo = DATA_DIR / "gold_modelo_predictivo.csv"

df_predictivo.to_csv(
    output_predictivo,
    index=False,
    encoding="utf-8-sig"
)

output_predictivo

WindowsPath('d:/INSTITUTO CONTINENTAL/4. CICLO - 4/BIG DATA APLICADA/PP2B_SATDPO/data/processed/gold_modelo_predictivo.csv')

## 3. Extraer dataset mensual de semáforo

Este archivo contiene la foto mensual por asesor: indicadores, niveles de cumplimiento, score y semáforo actual.


In [7]:
conn = get_connection()

query_mensual = """
select *
from gold.modelo_semaforo_asesor_mes
order by dni, periodo;
"""

df_mensual = pd.read_sql(query_mensual, conn)
conn.close()

df_mensual.shape

C:\Users\Steph\AppData\Local\Temp\ipykernel_6756\2683227670.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_mensual = pd.read_sql(query_mensual, conn)


(1680, 34)

In [8]:
df_mensual.head()

,dni,asesor,area,sede,fecha_ingreso,estado_trabajador,anio,mes,periodo,aprobacion_inmediata_pct,...,nc_tipificacion,alerta_aprobacion,alerta_horas_conexion,alerta_tmo,alerta_nps,alerta_calidad,alerta_tipificacion,score_final,semaforo_actual,tmo_original_bronze
0,40107223,María Rodríguez Morales,Gestión de Reportes,Los Olivos,2021-08-03,Cesado,2024,01,2024-01,45.0,...,69.44,Crítico,Crítico,Crítico,Adecuado,Crítico,Crítico,53.69,Rojo,6.87
1,40107223,María Rodríguez Morales,Gestión de Reportes,Los Olivos,2021-08-03,Cesado,2024,02,2024-02,55.0,...,59.38,Adecuado,Crítico,Crítico,Adecuado,Crítico,Crítico,69.06,Rojo,6.67
2,40107223,María Rodríguez Morales,Gestión de Reportes,Los Olivos,2021-08-03,Cesado,2024,03,2024-03,50.0,...,92.00,Adecuado,Crítico,Crítico,Adecuado,Crítico,Por mejorar,71.91,Ámbar,6.94
3,40107223,María Rodríguez Morales,Gestión de Reportes,Los Olivos,2021-08-03,Cesado,2024,04,2024-04,45.0,...,95.79,Crítico,Crítico,Crítico,Adecuado,Crítico,Por mejorar,65.96,Rojo,6.98
4,40107223,María Rodríguez Morales,Gestión de Reportes,Los Olivos,2021-08-03,Cesado,2024,05,2024-05,45.0,...,44.44,Crítico,Crítico,Crítico,Crítico,Crítico,Crítico,24.11,Rojo,6.84


In [9]:
if "semaforo_actual" in df_mensual.columns:
    display(df_mensual["semaforo_actual"].value_counts(dropna=False))

if "periodo" in df_mensual.columns:
    display(
        df_mensual.groupby("periodo")
        .size()
        .reset_index(name="registros")
        .sort_values("periodo")
    )

semaforo_actual
Rojo     1084
Ámbar     596
Name: count, dtype: int64

,periodo,registros
0,2024-01,60
1,2024-02,60
2,2024-03,60
3,2024-04,60
4,2024-05,60
5,2024-06,60
6,2024-07,60
7,2024-08,60
8,2024-09,60
9,2024-10,60


In [10]:
output_mensual = DATA_DIR / "gold_semaforo_asesor_mes.csv"

df_mensual.to_csv(
    output_mensual,
    index=False,
    encoding="utf-8-sig"
)

output_mensual

WindowsPath('d:/INSTITUTO CONTINENTAL/4. CICLO - 4/BIG DATA APLICADA/PP2B_SATDPO/data/processed/gold_semaforo_asesor_mes.csv')

## 4. Extraer puntos de mejora

Este archivo se usará para mostrar recomendaciones por asesor e indicador.


In [11]:
conn = get_connection()

query_mejora = """
select *
from gold.puntos_mejora_asesor
order by dni, periodo, prioridad_mejora;
"""

df_mejora = pd.read_sql(query_mejora, conn)
conn.close()

df_mejora.shape

C:\Users\Steph\AppData\Local\Temp\ipykernel_6756\4218963505.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_mejora = pd.read_sql(query_mejora, conn)


(10080, 12)

In [12]:
df_mejora.head()

,dni,asesor,anio,mes,periodo,indicador,valor_original,nivel_cumplimiento,nivel_alerta,peso_indicador,recomendacion,prioridad_mejora
0,40107223,María Rodríguez Morales,2024,01,2024-01,TMO,6.059500,0.00,Crítico,0.20,"Mejorar guion de atención, reducir tiempos mue...",1
1,40107223,María Rodríguez Morales,2024,01,2024-01,Aprobación inmediata,45.000000,0.00,Crítico,0.10,"Revisar criterios de aprobación, validación do...",2
2,40107223,María Rodríguez Morales,2024,01,2024-01,Horas conexión,79.397222,0.00,Crítico,0.10,"Controlar puntualidad, pausas, desconexiones y...",3
3,40107223,María Rodríguez Morales,2024,01,2024-01,Calidad,77.400000,37.00,Crítico,0.25,"Reforzar protocolo de atención, validación de ...",4
4,40107223,María Rodríguez Morales,2024,01,2024-01,Tipificación CRM,86.944444,69.44,Crítico,0.10,"Capacitar en registro correcto, codificación d...",5


In [13]:
output_mejora = DATA_DIR / "gold_puntos_mejora_asesor.csv"

df_mejora.to_csv(
    output_mejora,
    index=False,
    encoding="utf-8-sig"
)

output_mejora

WindowsPath('d:/INSTITUTO CONTINENTAL/4. CICLO - 4/BIG DATA APLICADA/PP2B_SATDPO/data/processed/gold_puntos_mejora_asesor.csv')

## 5. Resumen final de extracción

In [14]:
print("Archivos generados en:", DATA_DIR)
print("- Predictivo:", df_predictivo.shape, "->", output_predictivo.name)
print("- Mensual:", df_mensual.shape, "->", output_mensual.name)
print("- Mejora:", df_mejora.shape, "->", output_mejora.name)

Archivos generados en: d:\INSTITUTO CONTINENTAL\4. CICLO - 4\BIG DATA APLICADA\PP2B_SATDPO\data\processed
- Predictivo: (1620, 37) -> gold_modelo_predictivo.csv
- Mensual: (1680, 34) -> gold_semaforo_asesor_mes.csv
- Mejora: (10080, 12) -> gold_puntos_mejora_asesor.csv
